In [1]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.compose import ColumnTransformer
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("adult.csv",na_values="?")
df=df.drop_duplicates()
feat=df.drop(columns=['sex','race'])
feat=feat.dropna()
target = feat['income']
X = feat.drop(columns=['income'])
Y = target.map({">50K": 1, "<=50K": 0})
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)


In [3]:
num_cols = ["capital.loss", "fnlwgt"]
cat_cols = ["workclass","education","marital.status",
            "occupation","native.country"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])
models={"MLP":MLPClassifier(),"RF":RandomForestClassifier(),"SVC":SVC()}
pipelines={}
for name,model in models.items():
  pipelines[name]=Pipeline(steps=[('prosser',preprocessor),('model',model)])
print(pipelines)

{'MLP': Pipeline(steps=[('prosser',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['capital.loss', 'fnlwgt']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['workclass', 'education',
                                                   'marital.status',
                                                   'occupation',
                                                   'native.country'])])),
                ('model', MLPClassifier())]), 'RF': Pipeline(steps=[('prosser',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['capital.loss', 'fnlwgt']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'

In [ ]:
for name,pipe in pipelines.items():
  train_score=cross_val_score(pipe,X_train,Y_train,cv=3)
  print(name,":",train_score.mean())
"""
MLP : 0.8175521546182241
RF : 0.7826303347020032
SVC : 0.8265107212475634
"""

c:\Users\VAX\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
params={
    "model__hidden_layer_sizes": [(100,), (128,64),(128,64,32)],
    "model__max_iter": [500],
    "model__solver":['adam']
}
gcv=GridSearchCV(pipelines['MLP'],param_grid=params,cv=3)
gcv.fit(X_train,Y_train)
print(gcv.best_estimator_)
print(gcv.best_params_)
print(gcv.best_score_)


Pipeline(steps=[('prosser',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['capital.loss', 'fnlwgt']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['workclass', 'education',
                                                   'marital.status',
                                                   'occupation',
                                                   'native.country'])])),
                ('model', MLPClassifier(max_iter=500))])
{'model__hidden_layer_sizes': (100,), 'model__max_iter': 500, 'model__solver': 'adam'}
0.8115797768653312


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


In [ ]:
import joblib
joblib.dump(gcv.best_estimator_,'adult.pkl')

['adult.plk']